In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score
from sklearn.model_selection import StratifiedKFold
from pyod.models.deep_svdd import DeepSVDD
import sys
import scrapbook as sb

sys.path.append('..')
from utils import reshape_to_numpy, interpolate_missing_values, denoise_data, compute_spectrograms, time_avg_pooling

In [2]:
seed = 1
sampling_rate = 10
nperseg=64
noverlap=32
accel_cutoff = 0.4
accel_order = 30
gyro_cutoff = 0.6
gyro_order = 20
batch_size = 32
epochs = 100
hidden_neurons = [128, 64, 32]
n_splits = 5

In [3]:
# Parameters
seed = 9


In [4]:
rng = np.random.RandomState(seed)

In [5]:
data = pd.read_parquet("../data/GBG500.parquet")
data

,ride_id,time_index,ax,ay,az,rx,ry,rz
0,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,0,-2.512796,-9.385012,-1.053078,-0.009155,0.009155,-0.201416
1,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,100,-2.491268,-9.385012,-1.079390,-0.036621,0.027465,-0.155639
2,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,200,-2.534324,-9.382022,-1.030952,0.036621,0.027465,-0.073242
3,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,300,-2.488278,-9.383218,-1.091948,-0.045776,0.036621,-0.183105
4,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,400,-2.483494,-9.382022,-1.100320,-0.045776,0.027465,-0.192260
...,...,...,...,...,...,...,...,...
1529542,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241300,0.459862,-9.140430,-3.318900,1.556396,-0.091552,0.274658
1529543,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241400,0.455676,-9.140430,-3.321292,1.583861,-0.119018,0.274658
1529544,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241500,0.459862,-9.139832,-3.317704,1.583861,-0.109863,0.274658
1529545,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241600,0.456274,-9.142224,-3.321292,1.574707,-0.137329,0.283813


In [6]:
labels = pd.read_csv("../data/GBG500_labels.csv")
labels.columns = labels.columns.str.lower()
ride_order_df = pd.DataFrame({"ride_id": data["ride_id"].unique()})
labels_sorted = ride_order_df.merge(
    labels,
    on="ride_id",
    how="left"
)
labels_sorted

,ride_id,label
0,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,Safe
1,00d223cb7aecc9c0cc5871e32a6c45027a068eba07c572...,Reckless
2,02288a4aeca044203394e982e76b87818021dea2b34df9...,Safe
3,0236bdcf13d473ea24d97f5eaeea459f257dffac005694...,Bad weather
4,0238e5dd85143f1b6c59b200bc32f14361b8fc84c51a87...,Safe
...,...,...
495,fe5d7f84d692bbccad8bd566a3ad5c3108fa9f90acd671...,Safe
496,fe67169632306d4668b2affedef510df2cb43e2fb41220...,Safe
497,fee44667fdc8ab4995c702fc3bd36178de0b1ca2c64687...,Safe
498,ff79b2e945b93c2a5efc3a06365267b3a160e7849ba57d...,Safe


In [7]:
data_np = reshape_to_numpy(
    data,
    features = ["ax", "ay", "az", "rx", "ry", "rz"],
    max_timestamps = 4800
)

data_np_clean = interpolate_missing_values(
    data_np,
    method='linear',
    limit=None
)

data_np_clean = denoise_data(
    data=data_np_clean,
    accel_indices=[0, 1, 2],
    gyro_indices=[3, 4, 5],
    accel_cutoff=accel_cutoff,
    accel_order=accel_order,
    gyro_cutoff=gyro_cutoff,
    gyro_order=gyro_order,
)

In [8]:
# Compute spectrograms for all rides
spectrograms_array = compute_spectrograms(data_np_clean, sampling_rate, nperseg, noverlap)
spectrograms_array.shape

(500, 149, 6, 33)

In [9]:
# Aggregate spectrograms using time-averaged pooling
X_feat = time_avg_pooling(spectrograms_array)
X_feat.shape

(500, 198)

In [10]:
# 5-fold stratified CV with Deep SVDD
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=rng.randint(1000))
y_true = (labels_sorted['label'] == 'Reckless').astype(int).values
anomaly_scores = np.full(len(y_true), np.nan)

for fold, (train_idx, test_idx) in enumerate(skf.split(X_feat, y_true)):
    sc = StandardScaler()
    X_train = sc.fit_transform(X_feat[train_idx])
    X_test = sc.transform(X_feat[test_idx])

    np.random.seed(rng.randint(1000))
    model = DeepSVDD(
        n_features=X_train.shape[1],
        hidden_neurons=hidden_neurons,
        epochs=epochs,
        batch_size=batch_size,
        random_state=rng.randint(1000),
    )
    model.fit(X_train)
    anomaly_scores[test_idx] = model.decision_function(X_test)
    print(f"Fold {fold+1}/{n_splits} done")

Epoch 1/100, Loss: 3.7199578136205673
Epoch 2/100, Loss: 3.7182872965931892
Epoch 3/100, Loss: 3.5706899017095566
Epoch 4/100, Loss: 3.5669682696461678
Epoch 5/100, Loss: 3.2780843377113342
Epoch 6/100, Loss: 3.720730096101761
Epoch 7/100, Loss: 3.371754363179207
Epoch 8/100, Loss: 3.4919983819127083
Epoch 9/100, Loss: 3.367808520793915
Epoch 10/100, Loss: 3.4190922379493713
Epoch 11/100, Loss: 3.3853906095027924
Epoch 12/100, Loss: 3.137272484600544
Epoch 13/100, Loss: 3.2745786905288696
Epoch 14/100, Loss: 3.3826003298163414
Epoch 15/100, Loss: 3.444114252924919
Epoch 16/100, Loss: 3.6381419971585274


Epoch 17/100, Loss: 3.563033200800419
Epoch 18/100, Loss: 3.06746743619442
Epoch 19/100, Loss: 3.117598555982113
Epoch 20/100, Loss: 3.103740446269512
Epoch 21/100, Loss: 3.281360685825348
Epoch 22/100, Loss: 3.0922300294041634
Epoch 23/100, Loss: 3.1734316051006317
Epoch 24/100, Loss: 3.470989443361759
Epoch 25/100, Loss: 3.5510887801647186
Epoch 26/100, Loss: 4.033645816147327
Epoch 27/100, Loss: 3.1393388509750366
Epoch 28/100, Loss: 3.3487640097737312
Epoch 29/100, Loss: 3.375172294676304
Epoch 30/100, Loss: 2.895311616361141
Epoch 31/100, Loss: 2.8340402469038963
Epoch 32/100, Loss: 3.466525934636593
Epoch 33/100, Loss: 3.5088183358311653


Epoch 34/100, Loss: 3.467977650463581
Epoch 35/100, Loss: 3.3356440514326096
Epoch 36/100, Loss: 3.596568800508976
Epoch 37/100, Loss: 3.461627669632435
Epoch 38/100, Loss: 3.387589767575264
Epoch 39/100, Loss: 3.5587241873145103
Epoch 40/100, Loss: 3.287279672920704
Epoch 41/100, Loss: 3.245750941336155
Epoch 42/100, Loss: 3.221146784722805
Epoch 43/100, Loss: 3.4930632784962654
Epoch 44/100, Loss: 3.602897897362709
Epoch 45/100, Loss: 3.1814039796590805
Epoch 46/100, Loss: 2.8235175162553787


Epoch 47/100, Loss: 3.7491756677627563
Epoch 48/100, Loss: 3.3079160526394844
Epoch 49/100, Loss: 3.3020636811852455
Epoch 50/100, Loss: 3.2478305399417877
Epoch 51/100, Loss: 3.2379269152879715
Epoch 52/100, Loss: 4.428581342101097
Epoch 53/100, Loss: 3.289524368941784
Epoch 54/100, Loss: 3.2270707190036774
Epoch 55/100, Loss: 3.4859494492411613
Epoch 56/100, Loss: 3.530057266354561
Epoch 57/100, Loss: 3.5336447283625603
Epoch 58/100, Loss: 3.240125611424446
Epoch 59/100, Loss: 3.307643212378025
Epoch 60/100, Loss: 3.246477499604225
Epoch 61/100, Loss: 3.600582130253315
Epoch 62/100, Loss: 3.1843531727790833
Epoch 63/100, Loss: 3.1842210218310356


Epoch 64/100, Loss: 3.3326354697346687
Epoch 65/100, Loss: 3.577835887670517
Epoch 66/100, Loss: 3.2944493517279625
Epoch 67/100, Loss: 3.449589304625988
Epoch 68/100, Loss: 4.497655890882015
Epoch 69/100, Loss: 4.255186915397644
Epoch 70/100, Loss: 3.5978747829794884
Epoch 71/100, Loss: 3.3073591962456703
Epoch 72/100, Loss: 3.2414151579141617
Epoch 73/100, Loss: 3.570717342197895
Epoch 74/100, Loss: 3.2335276678204536
Epoch 75/100, Loss: 3.5532966554164886
Epoch 76/100, Loss: 3.247092120349407
Epoch 77/100, Loss: 3.6305154860019684
Epoch 78/100, Loss: 4.600147031247616
Epoch 79/100, Loss: 3.2902094200253487
Epoch 80/100, Loss: 3.400944374501705
Epoch 81/100, Loss: 3.1098847091197968


...

Epoch 27/100, Loss: 2.1641431227326393
Epoch 28/100, Loss: 1.9420833140611649
Epoch 29/100, Loss: 2.037737987935543
Epoch 30/100, Loss: 2.156177170574665
Epoch 31/100, Loss: 2.100159525871277
Epoch 32/100, Loss: 2.117270030081272
Epoch 33/100, Loss: 2.070302374660969
Epoch 34/100, Loss: 2.1987309604883194
Epoch 35/100, Loss: 2.1881012320518494
Epoch 36/100, Loss: 2.0700634717941284
Epoch 37/100, Loss: 2.1329702362418175
Epoch 38/100, Loss: 2.066109284758568
Epoch 39/100, Loss: 2.1122723296284676
Epoch 40/100, Loss: 2.1266877949237823
Epoch 41/100, Loss: 2.182592123746872
Epoch 42/100, Loss: 2.058465339243412
Epoch 43/100, Loss: 2.093743659555912
Epoch 44/100, Loss: 2.193540185689926


Epoch 45/100, Loss: 2.187866725027561
Epoch 46/100, Loss: 2.0570753291249275
Epoch 47/100, Loss: 2.177008792757988
Epoch 48/100, Loss: 2.2665006294846535
Epoch 49/100, Loss: 2.2005822733044624
Epoch 50/100, Loss: 2.1936435252428055
Epoch 51/100, Loss: 2.150972805917263
Epoch 52/100, Loss: 2.1326941400766373
Epoch 53/100, Loss: 2.1164422407746315
Epoch 54/100, Loss: 2.1277750059962273
Epoch 55/100, Loss: 2.1123837754130363
Epoch 56/100, Loss: 2.169858083128929
Epoch 57/100, Loss: 2.0621099323034286
Epoch 58/100, Loss: 2.3214476481080055
Epoch 59/100, Loss: 2.1921027824282646
Epoch 60/100, Loss: 2.2308334335684776
Epoch 61/100, Loss: 2.043168179690838


Epoch 62/100, Loss: 2.2022342458367348
Epoch 63/100, Loss: 2.2912934198975563
Epoch 64/100, Loss: 1.9520652890205383
Epoch 65/100, Loss: 2.1960667222738266
Epoch 66/100, Loss: 2.1337594985961914
Epoch 67/100, Loss: 2.1050184220075607
Epoch 68/100, Loss: 2.128612421452999
Epoch 69/100, Loss: 2.13801446557045
Epoch 70/100, Loss: 2.1798294112086296
Epoch 71/100, Loss: 2.1376819908618927
Epoch 72/100, Loss: 2.1615057587623596
Epoch 73/100, Loss: 2.1073620840907097
Epoch 74/100, Loss: 2.1844445392489433
Epoch 75/100, Loss: 2.231766700744629
Epoch 76/100, Loss: 2.2187138944864273
Epoch 77/100, Loss: 2.16849447786808
Epoch 78/100, Loss: 2.2457421123981476


Epoch 79/100, Loss: 2.1827515214681625
Epoch 80/100, Loss: 1.9967439696192741
Epoch 81/100, Loss: 2.2366557493805885
Epoch 82/100, Loss: 2.2089874744415283
Epoch 83/100, Loss: 2.1223436444997787
Epoch 84/100, Loss: 2.1899722665548325
Epoch 85/100, Loss: 2.0628085657954216
Epoch 86/100, Loss: 2.0650573670864105
Epoch 87/100, Loss: 2.2118449211120605
Epoch 88/100, Loss: 2.334926590323448
Epoch 89/100, Loss: 2.1566963642835617
Epoch 90/100, Loss: 2.0194173231720924
Epoch 91/100, Loss: 2.1508122235536575
Epoch 92/100, Loss: 2.0654619187116623
Epoch 93/100, Loss: 2.196831628680229
Epoch 94/100, Loss: 2.1326340585947037
Epoch 95/100, Loss: 2.243595838546753
Epoch 96/100, Loss: 2.1405412778258324


Epoch 97/100, Loss: 2.136247042566538
Epoch 98/100, Loss: 2.161115847527981
Epoch 99/100, Loss: 2.03836752474308
Epoch 100/100, Loss: 2.189760237932205
Fold 5/5 done


In [11]:
ap = average_precision_score(y_true, anomaly_scores)
print(f"Deep SVDD AP (5-fold CV) = {ap:.4f}")
sb.glue("GBG500_ap_spectral_deep_svdd", float(ap))

Deep SVDD AP (5-fold CV) = 0.5960
